# Repertoire growth per agent

How many distinct arguments does each agent accumulate over time, and does acquisition speed correlate with belief strength?

In [ ]:
from dataclasses import replace
from rankers import Config, run
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
cfg = Config(
    n                    = 2_000,
    k                    = 6,
    p_rewire             = 0.1,
    n_claims             = 200,
    llr_std              = 1.0,
    repertoire_seed_size = 5,
    emission_temp        = 1.0,
    n_steps              = 500,
    seed                 = 42,
)

N_TRACKED = 100

r = run(cfg, track_agents=N_TRACKED)
print(f"Elapsed : {r.elapsed_s:.3f} s")

rep     = r.repertoire_history   # (n_steps, N_TRACKED)
ids     = r.tracked_agents       # (N_TRACKED,)
final_b = r.final_beliefs[ids]   # final belief of each tracked agent
t       = np.arange(rep.shape[0])

print(f"Mean final repertoire : {rep[-1].mean():.1f} / {cfg.n_claims}  ({rep[-1].mean()/cfg.n_claims*100:.1f}%)")
print(f"Min / Max             : {rep[-1].min()} / {rep[-1].max()}")

## Trajectories + belief correlation

In [ ]:
vmax = np.abs(final_b).max()
norm = plt.Normalize(-vmax, vmax)
cmap = plt.cm.RdBu_r

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Repertoire growth — WS(N={cfg.n}, k={cfg.k}, p={cfg.p_rewire})  |  "
    f"pool M={cfg.n_claims}  seed={cfg.repertoire_seed_size}  "
    f"$\\beta$={cfg.emission_temp}",
    y=1.01,
)

# ── Left: trajectories coloured by final belief ───────────────────────────────
ax = axes[0]
for i in range(rep.shape[1]):
    ax.plot(t, rep[:, i], color=cmap(norm(final_b[i])), alpha=0.25, lw=0.8)
ax.plot(t, rep.mean(axis=1), color="black", lw=2.5, label="mean")
ax.axhline(cfg.repertoire_seed_size, ls=":",  color="gray", lw=1, label="seed size")
ax.axhline(cfg.n_claims,             ls="--", color="gray", lw=1, label=f"pool size ({cfg.n_claims})")
ax.set(xlabel="Step", ylabel="Claims in repertoire", title="Repertoire size over time")
ax.legend(fontsize=8)
fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
             label="Final belief $\\ell$", shrink=0.8)

# ── Right: final size vs final belief ────────────────────────────────────────
ax2 = axes[1]
sc = ax2.scatter(final_b, rep[-1], c=final_b, cmap=cmap, norm=norm,
                 edgecolors="white", linewidths=0.4, s=45, alpha=0.85)
ax2.axvline(0, ls="--", color="gray", lw=1, alpha=0.6)
ax2.set(xlabel="Final belief $\\ell$", ylabel="Final repertoire size",
        title="Repertoire size vs belief at $t_{\\mathrm{end}}$")
fig.colorbar(sc, ax=ax2, label="Final belief $\\ell$", shrink=0.8)

plt.tight_layout()
plt.show()

## Sensitivity: how emission temperature affects acquisition

In [ ]:
temps = [0.0, 0.5, 1.0, 2.0, 4.0]
cmap2 = plt.cm.viridis
colors2 = [cmap2(i / (len(temps) - 1)) for i in range(len(temps))]

fig, ax = plt.subplots(figsize=(9, 4))
for beta, col in zip(temps, colors2):
    r_b = run(replace(cfg, emission_temp=beta, seed=0), track_agents=N_TRACKED)
    mean_rep = r_b.repertoire_history.mean(axis=1)
    ax.plot(t, mean_rep, color=col, label=f"$\\beta$={beta}")

ax.axhline(cfg.n_claims, ls="--", color="gray", lw=1, label=f"pool ({cfg.n_claims})")
ax.set(xlabel="Step", ylabel="Mean repertoire size",
       title="Mean repertoire growth for different emission temperatures")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Sensitivity: rewiring probability

In [ ]:
p_values = [0.0, 0.01, 0.1, 0.3, 1.0]
cmap3    = plt.cm.plasma
colors3  = [cmap3(i / (len(p_values) - 1)) for i in range(len(p_values))]

fig, ax = plt.subplots(figsize=(9, 4))
for p, col in zip(p_values, colors3):
    r_p = run(replace(cfg, p_rewire=p, seed=0), track_agents=N_TRACKED)
    mean_rep = r_p.repertoire_history.mean(axis=1)
    ax.plot(t, mean_rep, color=col, label=f"p={p}")

ax.axhline(cfg.n_claims, ls="--", color="gray", lw=1, label=f"pool ({cfg.n_claims})")
ax.set(xlabel="Step", ylabel="Mean repertoire size",
       title="Mean repertoire growth for different rewiring probabilities")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()